# Unidad 2: Arquitectura de Apache Spark

**Tecnicatura en Datos – Procesamiento con Apache Spark**  
Unidad 2 de 10 — Duración estimada: 2:30 hs

> **Nota Databricks:** El objeto `spark` ya está disponible en el clúster. Las celdas de configuración muestran cómo inspeccionarlo.

---
## 1. ¿Qué pasa cuando ejecutamos código Spark?

Spark no es una caja negra. Antes de ejecutar cualquier operación, construye un plan, lo optimiza y lo distribuye. Entender este proceso es fundamental para diagnosticar problemas de rendimiento.

```
┌─────────────────────────────────────────────────────┐
│                   DRIVER PROGRAM                    │
│  (tu código Python / SparkSession)                  │
│   → construye el DAG                               │
│   → coordina la ejecución                          │
└──────────────────────┬──────────────────────────────┘
                       │ envía tasks
          ┌────────────▼────────────┐
          │    Cluster Manager      │  (YARN / Kubernetes / Standalone)
          └────────────┬────────────┘
                       │ asigna recursos
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
  ┌──────────┐  ┌──────────┐  ┌──────────┐
  │Executor 1│  │Executor 2│  │Executor N│
  │ tasks    │  │ tasks    │  │ tasks    │
  │ cache    │  │ cache    │  │ cache    │
  └──────────┘  └──────────┘  └──────────┘
```

---
## 2. El Driver Program

El **Driver** es el proceso principal. Es donde corre **tu código**.

Responsabilidades:
- Ejecutar el código del usuario
- Crear y gestionar la `SparkSession`
- Construir el DAG (plan de ejecución)
- Coordinar los Executors
- Recoger los resultados finales

> ⚠️ El Driver **no procesa datos masivos**: coordina. Si tu Driver consume demasiada memoria, generalmente es porque hiciste un `.collect()` de demasiados datos.

---
## 3. SparkSession — el punto de entrada

In [ ]:
# En Databricks: spark ya está disponible
# Para entorno local, usar:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder \
#     .appName("Unidad2_Arquitectura") \
#     .master("local[*]") \
#     .getOrCreate()

# Inspeccionar la SparkSession actual
print("=== SparkSession ===")
print(f"App Name  : {spark.sparkContext.appName}")
print(f"Spark ver : {spark.version}")
print(f"Master    : {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

**Resultado esperado (Databricks):**
```
=== SparkSession ===
App Name  : Databricks Shell
Spark ver : 3.x.x
Master    : spark://...
Default parallelism: 8   # varía según el clúster
```

---
## 4. Executors

Los **Executors** son los procesos que realmente procesan los datos. Corren en los nodos del clúster.

Cada Executor:
- Recibe **tasks** del Driver
- Procesa datos de su partición
- Almacena datos en memoria (caché)
- Reporta resultados al Driver

En Databricks, los Executors se configuran al crear el clúster (número de workers, cores por worker, RAM por worker).

In [ ]:
# Ver información de los Executors disponibles
sc = spark.sparkContext

print("=== Recursos del clúster ===")
print(f"Cores totales disponibles : {sc.defaultParallelism}")
print(f"Executors activos         : {len(sc.statusTracker().getExecutorInfos())}")

# Ver configuración clave de memoria
configs = [
    "spark.executor.memory",
    "spark.driver.memory",
    "spark.executor.cores",
]
print("\n=== Configuración de recursos ===")
for cfg in configs:
    try:
        print(f"  {cfg}: {spark.conf.get(cfg)}")
    except Exception:
        print(f"  {cfg}: (usando valor por defecto)")

---
## 5. DAG — Directed Acyclic Graph

Antes de ejecutar, Spark construye un **grafo acíclico dirigido** que representa el plan de ejecución.

```
Código del usuario:
   df.filter(...).groupBy(...).agg(...)

DAG construido:
   [Scan] → [Filter] → [GroupBy] → [Agg]
                          ↑
                     (shuffle aquí)
```

El DAG permite a Spark:
1. **Optimizar** el plan antes de ejecutar (Catalyst)
2. **Recuperarse** ante fallos (recomputar solo la partición perdida)
3. **Paralelizar** tareas independientes automáticamente

---
## 6. Jobs, Stages y Tasks

In [ ]:
# Ejemplo para ver Jobs, Stages y Tasks en acción
# Después de ejecutar esta celda, revisar la Spark UI (pestaña "Spark Jobs")

from pyspark.sql.functions import col, sum as spark_sum

# Creamos un DataFrame de ejemplo
data = [(i, i % 5, float(i * 10)) for i in range(1, 101)]
df = spark.createDataFrame(data, ["id", "categoria", "valor"])

# Esta operación generará:
#   1 Job (por la acción show)
#   2 Stages (uno por partición local, otro post-shuffle)
#   N Tasks (una por partición)
resultado = df.groupBy("categoria").agg(spark_sum("valor").alias("total"))
resultado.show()

print("\n✅ Revisar Spark UI para ver el Job, Stages y Tasks generados")

**Resultado esperado:**
```
+---------+-------+
|categoria|  total|
+---------+-------+
|        0| 1000.0|
|        1| 1020.0|
|        2| 1040.0|
|        3| 1060.0|
|        4| 1080.0|
+---------+-------+
```

### Conceptos clave

| Concepto | Definición | Disparado por |
|---|---|---|
| **Job** | Unidad completa de trabajo | Una acción (`show`, `count`, `collect`) |
| **Stage** | Grupo de tasks sin shuffle | Separados por operaciones shuffle |
| **Task** | Unidad mínima de trabajo | Una por partición por Stage |

---
## 7. Lazy Evaluation — evaluación diferida

In [ ]:
import time

print("Definiendo transformaciones...")
t0 = time.time()

# Estas líneas NO ejecutan nada: solo construyen el plan
rdd_base = sc.parallelize(range(1, 1_000_001))
rdd_filtrado = rdd_base.filter(lambda x: x % 2 == 0)
rdd_mapeado  = rdd_filtrado.map(lambda x: x ** 2)

t1 = time.time()
print(f"Tiempo en definir transformaciones: {t1-t0:.4f}s  ← casi 0 (no ejecutó nada)")

# Esta línea SÍ ejecuta todo el plan
print("\nDisparando acción count()...")
t2 = time.time()
resultado = rdd_mapeado.count()
t3 = time.time()

print(f"Resultado: {resultado}")
print(f"Tiempo en ejecutar: {t3-t2:.4f}s  ← aquí ocurrió el procesamiento real")

**Resultado esperado:**
```
Definiendo transformaciones...
Tiempo en definir transformaciones: 0.0003s  ← casi 0 (no ejecutó nada)

Disparando acción count()...
Resultado: 500000
Tiempo en ejecutar: 2.1s  ← aquí ocurrió el procesamiento real
```

> La **Lazy Evaluation** permite a Spark ver el plan completo antes de ejecutar y optimizarlo globalmente. Si Spark ejecutara cada transformación inmediatamente, no podría aplicar optimizaciones como el predicate pushdown.

---
## 8. Modos de ejecución

| Modo | Dónde corre el Driver | Uso típico |
|---|---|---|
| **Local** | En tu máquina, un hilo | Desarrollo y testing |
| **Local[N]** | En tu máquina, N hilos | Testing con paralelismo |
| **Client** | En tu máquina, workers remotos | Desarrollo interactivo |
| **Cluster** | En el clúster (un worker) | Producción |

En **Databricks**, el modo es siempre Cluster: el Driver corre en un nodo dedicado del clúster.

In [ ]:
# Verificar el modo de ejecución actual
master = sc.master
print(f"Master actual: {master}")

if master.startswith("local"):
    print("Modo: LOCAL — ideal para desarrollo")
elif master.startswith("spark://"):
    print("Modo: CLUSTER STANDALONE")
elif "yarn" in master:
    print("Modo: YARN (Hadoop)")
else:
    print(f"Modo: {master}")

---
## 9. Caso de uso: procesamiento batch bancario

Un pipeline típico en producción sigue este flujo:

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg

# Simulación de un pipeline batch nocturno
# En producción: spark.read.parquet("/mnt/datalake/transacciones/")
transacciones = spark.createDataFrame([
    ("T001", "CL001", "AR", 1500.0, "2024-01-10"),
    ("T002", "CL002", "MX",  800.0, "2024-01-10"),
    ("T003", "CL001", "AR", 2200.0, "2024-01-10"),
    ("T004", "CL003", "CL",  450.0, "2024-01-10"),
    ("T005", "CL002", "MX",  300.0, "2024-01-10"),
    ("T006", "CL001", "AR",  900.0, "2024-01-10"),
], ["tx_id", "cliente_id", "pais", "monto", "fecha"])

# 1. Capa Bronze → Silver: filtrar y agregar
resumen = (
    transacciones
    .filter(col("monto") > 0)                         # validación básica
    .groupBy("cliente_id", "pais")
    .agg(
        spark_sum("monto").alias("monto_total"),
        count("tx_id").alias("num_transacciones"),
        avg("monto").alias("monto_promedio"),
    )
    .orderBy(col("monto_total").desc())
)

print("=== Resumen por cliente (capa Silver) ===")
resumen.show()

# 2. En producción se escribiría a Delta Lake:
# resumen.write.format("delta").mode("overwrite").save("/mnt/datalake/silver/resumen_clientes")

**Resultado esperado:**
```
=== Resumen por cliente (capa Silver) ===
+----------+----+-----------+-----------------+--------------+
|cliente_id|pais|monto_total|num_transacciones|monto_promedio|
+----------+----+-----------+-----------------+--------------+
|     CL001|  AR|     4600.0|                3|    1533.33...|
|     CL002|  MX|     1100.0|                2|         550.0|
|     CL003|  CL|      450.0|                1|         450.0|
+----------+----+-----------+-----------------+--------------+
```

---
## 10. Práctica: identificar Jobs y Stages en la Spark UI

Ejecuta la siguiente celda y observa los Jobs generados en la pestaña **Spark** de Databricks:

In [ ]:
# Operación 1: sin shuffle → 1 Stage
print("=== Operación SIN shuffle ===")
df1 = spark.range(1, 10001).filter(col("id") % 2 == 0)
print(f"Pares encontrados: {df1.count()}")  # Job 1: 1 Stage

# Operación 2: con shuffle → 2 Stages
print("\n=== Operación CON shuffle (groupBy) ===")
df2 = spark.range(1, 10001).withColumn("grupo", col("id") % 4)
print(df2.groupBy("grupo").count().orderBy("grupo").collect())  # Job 2: 2+ Stages

print("\n✅ Inspeccionar la Spark UI para ver la diferencia de Stages entre Job 1 y Job 2")

---
## 11. Preguntas de revisión

1. ¿Cuál es la diferencia entre el Driver y un Executor?
2. ¿Por qué el Driver no debería procesar grandes volúmenes de datos?
3. ¿Qué es un DAG y qué ventaja da la Lazy Evaluation?
4. ¿Cuándo se divide un Job en múltiples Stages?
5. ¿Cuántas Tasks se generan en un Stage con 8 particiones?

---
**Próxima unidad:** RDDs — Resilient Distributed Datasets